# CSE301 Machine Learning Capstone — Regression Track (Algorithms 1–5)
## Review 1 | HeartFailureDataset.csv

This is **one notebook with five separate algorithm sections and separate code cells**:
1. Linear Regression
2. Ridge Regression
3. Lasso Regression
4. ElasticNet Regression
5. Polynomial Regression

Each algorithm has its own training and evaluation cell. Shared preprocessing is done once so all five models use the same cleaned data and test split.

**Target note:** `cardio` is a binary classification outcome, so this notebook uses `ap_hi` (systolic blood pressure) as the regression target. Confirm with your instructor that this is the assigned regression target.

## 1. Imports and configuration

In [ ]:
# If needed, install dependencies in a separate cell:
# %pip install pandas numpy matplotlib seaborn scikit-learn

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.base import clone

RANDOM_STATE = 42
DATA_PATH = "HeartFailureDataset.csv"
TARGET_COLUMN = "ap_hi"
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 2. Load and audit the dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())
print("\nData types:")
display(df.dtypes.to_frame("dtype"))
print("\nMissing values:")
display(df.isna().sum().to_frame("missing_count"))
print("\nDuplicate rows:", df.duplicated().sum())

if "cardio" in df.columns:
    print("\nCardio distribution (context only):")
    display(df["cardio"].value_counts(dropna=False).to_frame("count"))

## 3. Cleaning and feature engineering

In [ ]:
data = df.copy().drop_duplicates().reset_index(drop=True)

if "age" in data.columns:
    data["age_years"] = data["age"] / 365.25
    data = data.drop(columns=["age"])

if {"height", "weight"}.issubset(data.columns):
    data["bmi"] = data["weight"] / ((data["height"] / 100) ** 2)
    data = data.drop(columns=["height", "weight"])

if TARGET_COLUMN not in data.columns:
    raise ValueError(f"Target '{TARGET_COLUMN}' not found. Columns: {data.columns.tolist()}")

y = data[TARGET_COLUMN].copy()
excluded = [c for c in ["id", "cardio", TARGET_COLUMN] if c in data.columns]
X = data.drop(columns=excluded).copy()

print("Target:", TARGET_COLUMN)
display(y.describe().to_frame("target_summary"))
print("Predictor columns:", X.columns.tolist())
print("Predictor shape:", X.shape)

## 4. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(y, bins=40, kde=True)
plt.title(f"Target Distribution: {TARGET_COLUMN}")
plt.xlabel(TARGET_COLUMN)
plt.tight_layout()
plt.show()

X.hist(figsize=(16, 12), bins=30, edgecolor="black")
plt.suptitle("Predictor Distributions", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
eda = pd.concat([X.select_dtypes(include=np.number), y.rename(TARGET_COLUMN)], axis=1)
plt.figure(figsize=(12, 8))
sns.heatmap(eda.corr(), cmap="coolwarm", center=0, annot=True, fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
scatter_cols = X.select_dtypes(include=np.number).columns[:4]
fig, axes = plt.subplots(1, len(scatter_cols), figsize=(5*len(scatter_cols), 4))
axes = np.array(axes).reshape(-1)
for ax, col in zip(axes, scatter_cols):
    sns.scatterplot(x=X[col], y=y, s=12, alpha=0.35, ax=ax)
    ax.set_title(f"{col} vs {TARGET_COLUMN}")
    ax.set_xlabel(col)
    ax.set_ylabel(TARGET_COLUMN)
plt.tight_layout()
plt.show()

## 5. Train/test split

In [ ]:
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("Training rows:", len(X_train_raw))
print("Test rows:", len(X_test_raw))

## 6. Missing values and outlier detection

In [ ]:
categorical_cols = [
    c for c in ["gender", "cholesterol", "gluc", "smoke", "alco", "active"]
    if c in X_train_raw.columns
]
numeric_cols = X_train_raw.select_dtypes(include=np.number).columns.tolist()
continuous_cols = [c for c in numeric_cols if c not in categorical_cols]

imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(
    imputer.fit_transform(X_train_raw),
    columns=X_train_raw.columns, index=X_train_raw.index
)
X_test_imp = pd.DataFrame(
    imputer.transform(X_test_raw),
    columns=X_test_raw.columns, index=X_test_raw.index
)

def get_iqr_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5*iqr, q3 + 1.5*iqr

def plot_boxplots(frame, columns, title, max_cols=4):
    if not columns:
        print("No columns available for boxplots.")
        return
    ncols = min(max_cols, len(columns))
    nrows = int(np.ceil(len(columns) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3.5*nrows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, columns):
        sns.boxplot(y=frame[col], ax=ax)
        ax.set_title(col)
    for ax in axes[len(columns):]:
        ax.set_visible(False)
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

print("Missing values after imputation:", int(X_train_imp.isna().sum().sum()))
plot_boxplots(X_train_imp, continuous_cols, "Predictor Boxplots — Before Outlier Treatment")
plot_boxplots(pd.DataFrame({TARGET_COLUMN: y_train_raw}), [TARGET_COLUMN], "Target Boxplot — Before Outlier Treatment")

outlier_rows = []
for col in continuous_cols:
    lo, hi = get_iqr_bounds(X_train_imp[col])
    outlier_rows.append({
        "feature": col, "lower_bound": lo, "upper_bound": hi,
        "outlier_count_train": int(((X_train_imp[col] < lo) | (X_train_imp[col] > hi)).sum())
    })
display(pd.DataFrame(outlier_rows))

## 7. Outlier treatment and boxplots after treatment

In [ ]:
X_train_clean = X_train_imp.copy()
X_test_clean = X_test_imp.copy()

for col in continuous_cols:
    lo, hi = get_iqr_bounds(X_train_imp[col])
    X_train_clean[col] = X_train_clean[col].clip(lo, hi)
    X_test_clean[col] = X_test_clean[col].clip(lo, hi)

target_lo, target_hi = get_iqr_bounds(y_train_raw)
y_train = y_train_raw.clip(target_lo, target_hi)
y_test = y_test_raw.clip(target_lo, target_hi)

plot_boxplots(X_train_clean, continuous_cols, "Predictor Boxplots — After Outlier Treatment")
plot_boxplots(pd.DataFrame({TARGET_COLUMN: y_train}), [TARGET_COLUMN], "Target Boxplot — After Outlier Treatment")
print("Target clipping bounds:", (target_lo, target_hi))

## 8. Feature scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_clean),
    columns=X_train_clean.columns, index=X_train_clean.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_clean),
    columns=X_test_clean.columns, index=X_test_clean.index
)
print("Scaled training shape:", X_train_scaled.shape)

# Algorithm 1 — Linear Regression

Baseline model. Its coefficients can be interpreted as the expected target change for a one-standard-deviation change in a scaled predictor, holding other predictors constant.

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train)
linear_pred = linear_model.predict(X_test_scaled)

linear_metrics = {
    "Algorithm": "Linear Regression",
    "R2": r2_score(y_test, linear_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, linear_pred)),
    "MAE": mean_absolute_error(y_test, linear_pred)
}
display(pd.DataFrame([linear_metrics]).round(4))

In [ ]:
linear_coef = pd.DataFrame({
    "Feature": X_train_scaled.columns,
    "Coefficient": linear_model.coef_
}).sort_values("Coefficient", key=lambda s: s.abs(), ascending=False)
print("Intercept:", linear_model.intercept_)
display(linear_coef)

# Algorithm 2 — Ridge Regression

Ridge applies L2 regularisation. GridSearchCV selects `alpha` using five-fold cross-validation.

In [ ]:
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_train)
ridge_pred = ridge_model.predict(X_test_scaled)

ridge_metrics = {
    "Algorithm": "Ridge Regression",
    "R2": r2_score(y_test, ridge_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, ridge_pred)),
    "MAE": mean_absolute_error(y_test, ridge_pred)
}
display(pd.DataFrame([ridge_metrics]).round(4))

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
ridge_search = GridSearchCV(
    Ridge(),
    param_grid={"alpha": [0.01, 0.1, 1.0, 10.0, 100.0]},
    scoring="r2", cv=cv, n_jobs=-1
)
ridge_search.fit(X_train_scaled, y_train)
ridge_tuned_pred = ridge_search.best_estimator_.predict(X_test_scaled)
display(pd.DataFrame([{
    "Best Parameters": str(ridge_search.best_params_),
    "Best CV R2": ridge_search.best_score_,
    "Test R2": r2_score(y_test, ridge_tuned_pred),
    "Test RMSE": np.sqrt(mean_squared_error(y_test, ridge_tuned_pred)),
    "Test MAE": mean_absolute_error(y_test, ridge_tuned_pred)
}]).round(4))

# Algorithm 3 — Lasso Regression

Lasso applies L1 regularisation and may shrink some coefficients to zero. GridSearchCV tunes `alpha`.

In [ ]:
lasso_model = Lasso(alpha=0.1, max_iter=20000, random_state=RANDOM_STATE)
lasso_model.fit(X_train_scaled, y_train)
lasso_pred = lasso_model.predict(X_test_scaled)

lasso_metrics = {
    "Algorithm": "Lasso Regression",
    "R2": r2_score(y_test, lasso_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, lasso_pred)),
    "MAE": mean_absolute_error(y_test, lasso_pred)
}
display(pd.DataFrame([lasso_metrics]).round(4))

In [ ]:
lasso_search = GridSearchCV(
    Lasso(max_iter=20000, random_state=RANDOM_STATE),
    param_grid={"alpha": [0.001, 0.01, 0.1, 1.0, 10.0]},
    scoring="r2", cv=cv, n_jobs=-1
)
lasso_search.fit(X_train_scaled, y_train)
lasso_tuned_pred = lasso_search.best_estimator_.predict(X_test_scaled)
display(pd.DataFrame([{
    "Best Parameters": str(lasso_search.best_params_),
    "Best CV R2": lasso_search.best_score_,
    "Test R2": r2_score(y_test, lasso_tuned_pred),
    "Test RMSE": np.sqrt(mean_squared_error(y_test, lasso_tuned_pred)),
    "Test MAE": mean_absolute_error(y_test, lasso_tuned_pred)
}]).round(4))

# Algorithm 4 — ElasticNet Regression

ElasticNet combines L1 and L2 regularisation. GridSearchCV tunes both `alpha` and `l1_ratio`.

In [ ]:
elastic_model = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=20000, random_state=RANDOM_STATE)
elastic_model.fit(X_train_scaled, y_train)
elastic_pred = elastic_model.predict(X_test_scaled)

elastic_metrics = {
    "Algorithm": "ElasticNet Regression",
    "R2": r2_score(y_test, elastic_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, elastic_pred)),
    "MAE": mean_absolute_error(y_test, elastic_pred)
}
display(pd.DataFrame([elastic_metrics]).round(4))

In [ ]:
elastic_search = GridSearchCV(
    ElasticNet(max_iter=20000, random_state=RANDOM_STATE),
    param_grid={
        "alpha": [0.001, 0.01, 0.1, 1.0],
        "l1_ratio": [0.2, 0.5, 0.8]
    },
    scoring="r2", cv=cv, n_jobs=-1
)
elastic_search.fit(X_train_scaled, y_train)
elastic_tuned_pred = elastic_search.best_estimator_.predict(X_test_scaled)
display(pd.DataFrame([{
    "Best Parameters": str(elastic_search.best_params_),
    "Best CV R2": elastic_search.best_score_,
    "Test R2": r2_score(y_test, elastic_tuned_pred),
    "Test RMSE": np.sqrt(mean_squared_error(y_test, elastic_tuned_pred)),
    "Test MAE": mean_absolute_error(y_test, elastic_tuned_pred)
}]).round(4))

# Algorithm 5 — Polynomial Regression

PolynomialFeatures expands predictors into degree-2 terms, then Linear Regression is trained on the expanded features. The polynomial scaler is fitted as part of the pipeline.

In [ ]:
polynomial_model = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scale_poly", StandardScaler()),
    ("regressor", LinearRegression())
])
polynomial_model.fit(X_train_scaled, y_train)
polynomial_pred = polynomial_model.predict(X_test_scaled)

polynomial_metrics = {
    "Algorithm": "Polynomial Regression (degree=2)",
    "R2": r2_score(y_test, polynomial_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, polynomial_pred)),
    "MAE": mean_absolute_error(y_test, polynomial_pred)
}
display(pd.DataFrame([polynomial_metrics]).round(4))

In [ ]:
degree_results = []
for degree in [1, 2, 3]:
    candidate = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("scale_poly", StandardScaler()),
        ("regressor", LinearRegression())
    ])
    candidate.fit(X_train_scaled, y_train)
    pred = candidate.predict(X_test_scaled)
    degree_results.append({
        "Polynomial Degree": degree,
        "R2": r2_score(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "MAE": mean_absolute_error(y_test, pred)
    })
display(pd.DataFrame(degree_results).sort_values("R2", ascending=False).round(4))

## 9. Final comparison table — all five algorithms

In [ ]:
comparison = pd.DataFrame([
    linear_metrics,
    ridge_metrics,
    lasso_metrics,
    elastic_metrics,
    polynomial_metrics
]).sort_values("R2", ascending=False).reset_index(drop=True)
display(comparison.round(4))

## 10. Five-fold cross-validation for the two best baseline models

In [ ]:
model_map = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.1, max_iter=20000, random_state=RANDOM_STATE),
    "ElasticNet Regression": ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=20000, random_state=RANDOM_STATE),
    "Polynomial Regression (degree=2)": Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scale_poly", StandardScaler()),
        ("regressor", LinearRegression())
    ])
}
top_names = comparison.head(2)["Algorithm"].tolist()
cv_rows = []
for name in top_names:
    scores = cross_val_score(model_map[name], X_train_scaled, y_train, scoring="r2", cv=cv, n_jobs=-1)
    cv_rows.append({"Algorithm": name, "5-Fold CV R2 Mean": scores.mean(), "5-Fold CV R2 Std": scores.std()})
display(pd.DataFrame(cv_rows).round(4))

## 11. Predicted-versus-actual and residual plots for the best model

In [ ]:
prediction_map = {
    "Linear Regression": linear_pred,
    "Ridge Regression": ridge_pred,
    "Lasso Regression": lasso_pred,
    "ElasticNet Regression": elastic_pred,
    "Polynomial Regression (degree=2)": polynomial_pred
}
best_name = comparison.iloc[0]["Algorithm"]
best_pred = prediction_map[best_name]
residuals = y_test - best_pred
print("Best baseline model by test R2:", best_name)

plt.figure(figsize=(7, 5))
sns.scatterplot(x=y_test, y=best_pred, alpha=0.45, s=18)
low, high = min(y_test.min(), best_pred.min()), max(y_test.max(), best_pred.max())
plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Actual target")
plt.ylabel("Predicted target")
plt.title(f"Predicted vs Actual — {best_name}")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
sns.scatterplot(x=best_pred, y=residuals, alpha=0.45, s=18)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted target")
plt.ylabel("Residual (actual − predicted)")
plt.title(f"Residual Plot — {best_name}")
plt.tight_layout()
plt.show()

## 12. Review 1 conclusion prompts

Use the actual results produced by this notebook and write your own observations:
1. Which model has the highest R²?
2. Which model has the lowest RMSE and MAE?
3. Did hyperparameter tuning improve Ridge, Lasso, or ElasticNet?
4. What do the residual and predicted-vs-actual plots show?
5. Why are the scaler and IQR bounds fitted using training data only?

**Important:** Confirm `ap_hi` as the regression target with your instructor before presentation. Do not use `cardio` as a regression target because it is binary.